# Notebook 2 · Essentials: How LangChain Works

### Real messages, the real agent loop, still zero cloud

Notebook 1 ended with a hand-rolled loop and a list of everything it was missing. Now we meet the real framework. Two promises:

- Every code cell runs the actual LangChain agent loop, with the real message objects.
- Nothing needs an API key. We swap the live model for a scripted stand-in, so every run is identical and free. One line turns it into a real Bedrock call, and we show that line each time.

**Setup once, then everything below just works.**

```bash
pip install langchain langgraph langchain-aws
```

### The teaching trick: a scripted model

A real chat model is non-deterministic and needs credentials. Neither is good for learning the mechanics. So we build a `ScriptedChatModel`: a real LangChain model whose replies are a fixed list you hand it. The framework treats it exactly like a Bedrock model, so the loop, the message types, and the tool calls are all genuine. Only the words are scripted.

> **What runs next** import the real LangChain pieces, then define `ScriptedChatModel`. Run this cell first, every later cell depends on it.
> **Python construct** subclassing, a class attribute used as a cursor (`idx`), `object.__setattr__` to mutate a field on a strict model.
> **LLM concept** a chat model is an object with an `invoke` method that maps messages to a message. That is the whole interface.

In [ ]:
# %pip install langchain langgraph langchain-aws

from typing import Any, List
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain.agents import create_agent
from langchain.tools import tool


class ScriptedChatModel(BaseChatModel):
    '''A real LangChain chat model that replays a fixed list of AIMessages.
    It supports bind_tools, so create_agent drives it exactly like a live model.
    Teaching only: deterministic and free. Swap for ChatBedrockConverse in production.
    '''
    responses: List[Any]
    idx: int = 0

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = self.responses[min(self.idx, len(self.responses) - 1)]
        object.__setattr__(self, "idx", self.idx + 1)
        return ChatResult(generations=[ChatGeneration(message=reply)])

    def bind_tools(self, tools, **kwargs):
        return self  # a live model would attach tool schemas here


print("ScriptedChatModel ready. The framework code below is 100% real.")

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 11.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.1/671.1 kB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.2/566.2 kB 22.6 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 640.7/640.7 kB 31.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [langchain]20 [langgraph]aws]]lt]
Note: you may need to restart the kernel to use updated packages.
ScriptedChatModel ready. The framework code below is 100% real.


> **What just happened** you now have a stand-in that plugs into the real agent machinery. Wherever you see `ScriptedChatModel(...)`, picture a live model. The production swap is one line:

```python
# Reference (run in an AWS environment with Bedrock access):
from langchain_aws import ChatBedrockConverse
model = ChatBedrockConverse(
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name="us-east-1",
    temperature=0,
)
```

---
## 1. Messages are the currency

LangChain does not pass raw strings between steps. It passes typed messages. Four types cover almost everything:

| Message | Who it is from | Role in the loop |
|---------|----------------|------------------|
| `SystemMessage` | you, the developer | standing instructions, set once |
| `HumanMessage` | the user | the request |
| `AIMessage` | the model | either a final answer or a request to call tools |
| `ToolMessage` | your code | the result of a tool the model asked for |

The agent state is just a growing list of these. The loop reads the list, appends to it, and reads again.

```mermaid
flowchart LR
    S[SystemMessage] --> L[messages list]
    H[HumanMessage] --> L
    L --> AI[AIMessage]
    AI --> TM[ToolMessage]
    TM --> L
```

### 1a. Build and inspect a message list

> **What runs next** construct one of each message type and print its type and content.
> **Python construct** instantiating classes, reading attributes.
> **Language concept** roles give a flat transcript structure. The model behaves differently depending on which role said what.

In [3]:
conversation = [
    SystemMessage(content="You are TravelMind, an airline support agent."),
    HumanMessage(content="Is my flight JX48Q2 cancelled?"),
    AIMessage(content="", tool_calls=[{"name": "lookup_pnr", "args": {"pnr": "JX48Q2"}, "id": "t1", "type": "tool_call"}]),
    ToolMessage(content="JX48Q2: BLR-DEL cancelled, Rao, Gold tier", tool_call_id="t1"),
    AIMessage(content="Yes. JX48Q2 on BLR-DEL is cancelled. You are Gold tier, so rebooking is free."),
]

for m in conversation:
    label = type(m).__name__
    if getattr(m, "tool_calls", None):
        print(f"{label:14} -> asks for tool: {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})")
    else:
        print(f"{label:14} -> {m.content}")

SystemMessage  -> You are TravelMind, an airline support agent.
HumanMessage   -> Is my flight JX48Q2 cancelled?
AIMessage      -> asks for tool: lookup_pnr({'pnr': 'JX48Q2'})
ToolMessage    -> JX48Q2: BLR-DEL cancelled, Rao, Gold tier
AIMessage      -> Yes. JX48Q2 on BLR-DEL is cancelled. You are Gold tier, so rebooking is free.


> **What just happened** this is a full agent turn frozen in place. Notice the third message: an `AIMessage` with empty text and a `tool_calls` field. That is the model saying "do not answer yet, call this tool first." The `ToolMessage` carries the answer back, keyed to the call id. The final `AIMessage` is the reply the user sees.

> **Skeptic's corner** why typed messages instead of a plain string blob? Because the loop must tell "the model wants a tool" apart from "the model answered". A string cannot carry that distinction cleanly. The types are the contract.

---
## 2. The agent loop, for real

`create_agent` builds the loop you hand-rolled in Notebook 1, and returns a runnable graph. You call it with a messages dict, it returns the full message list including everything that happened.

```mermaid
flowchart TD
    IN["invoke({messages: [...]})"] --> MOD[model node]
    MOD -->|AIMessage has tool_calls| TOO[tools node]
    TOO --> MOD
    MOD -->|no tool_calls| OUT[return full message list]
```

Start with the simplest possible agent: no tools. The loop runs once, the model answers, done.

### 2a. A bare agent (no tools)

> **What runs next** hand `create_agent` a scripted model and an empty tools list, then invoke it.
> **Python construct** a dict as the input payload, list indexing with `[-1]` for the last message.
> **LLM concept** with no tools the loop cannot branch. It is one model call wrapped in framework structure.

In [4]:
model = ScriptedChatModel(responses=[
    AIMessage(content="I help with bookings, cancellations, and rebooking. What is your PNR?"),
])

agent = create_agent(model, tools=[], system_prompt="You are TravelMind, an airline support agent.")

result = agent.invoke({"messages": [{"role": "user", "content": "Hi, can you help me?"}]})

print("messages in the final state:", len(result["messages"]))
for m in result["messages"]:
    print(" -", type(m).__name__, "|", m.content)

print("\nreply to the user:", result["messages"][-1].content)

messages in the final state: 2
 - HumanMessage | Hi, can you help me?
 - AIMessage | I help with bookings, cancellations, and rebooking. What is your PNR?

reply to the user: I help with bookings, cancellations, and rebooking. What is your PNR?


> **What just happened** two messages: the human turn and the model reply. No tool node was ever reached. This is the true baseline. Everything from here adds exactly one capability at a time.

```python
# Reference: the only change for a real model is the constructor.
model = ChatBedrockConverse(model="us.anthropic.claude-haiku-4-5-20251001-v1:0", region_name="us-east-1")
```

---
## 3. Give it a tool, watch the loop turn

Now the interesting part. Add one tool. When the model asks for it, the framework runs it, feeds the result back, and calls the model again. This is the loop earning its keep.

```mermaid
flowchart TD
    U[user: status of JX48Q2?] --> M1[model: call lookup_pnr]
    M1 --> T[tools node runs lookup_pnr]
    T --> M2[model reads result, writes answer]
    M2 --> DONE[final answer to user]
```

> **What runs next** define a real tool with `@tool`, script the model to call it and then answer, and print every message the loop produced.
> **Python construct** a decorator (`@tool`) that turns a function plus its docstring into a tool the model can see.
> **LLM concept** tool calling. The model does not run code. It emits a name and arguments, and your side runs it.
> **Language concept** grounding: the final answer is anchored to a real record, not to what the model guessed.

In [5]:
@tool
def lookup_pnr(pnr: str) -> str:
    '''Return the booking status for a passenger name record (PNR).'''
    records = {"JX48Q2": "BLR-DEL cancelled, Rao, Gold tier"}
    return records.get(pnr, "PNR not found")


model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "lookup_pnr", "args": {"pnr": "JX48Q2"}, "id": "t1", "type": "tool_call"}]),
    AIMessage(content="JX48Q2 on BLR-DEL is cancelled. You are Gold tier, so a rebook carries no fee."),
])

agent = create_agent(model, tools=[lookup_pnr], system_prompt="You are TravelMind.")
result = agent.invoke({"messages": [{"role": "user", "content": "Is JX48Q2 cancelled?"}]})

for m in result["messages"]:
    label = type(m).__name__
    if getattr(m, "tool_calls", None):
        print(f"{label:14} -> calls {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})")
    elif label == "ToolMessage":
        print(f"{label:14} -> {m.content}")
    else:
        print(f"{label:14} -> {m.content}")

HumanMessage   -> Is JX48Q2 cancelled?
AIMessage      -> calls lookup_pnr({'pnr': 'JX48Q2'})
ToolMessage    -> BLR-DEL cancelled, Rao, Gold tier
AIMessage      -> JX48Q2 on BLR-DEL is cancelled. You are Gold tier, so a rebook carries no fee.


> **What just happened** four messages, and they line up exactly with the loop diagram: human asks, model requests the tool, the tool answers, the model writes the grounded reply. Compare this with your hand-rolled `run_loop` from Notebook 1. Same shape, but message history, tool dispatch, and the stop condition are now the framework's job, not yours.

> **Gotcha** the tool docstring is not a comment, it is the interface the model reads to decide when to call the tool. A vague docstring produces a model that calls the wrong tool. Treat docstrings as production code. Notebook 3 makes this concrete.

---
## 4. Streaming: see the steps as they happen

`invoke` returns only the final state. For a live agent you usually want to show progress as it goes. `stream` yields each step as it completes. This matters for user experience and for debugging a loop that stalls.

```mermaid
flowchart LR
    S["agent.stream(..., stream_mode='updates')"] --> E1[update from model node]
    E1 --> E2[update from tools node]
    E2 --> E3[update from model node]
    E3 --> F[loop ends]
```

> **What runs next** run the same tool agent with `stream` and print which node produced each update.
> **Python construct** iterating a generator with a `for` loop.
> **LLM concept** the loop is a sequence of node updates. Streaming exposes that sequence instead of hiding it.

In [6]:
model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "lookup_pnr", "args": {"pnr": "JX48Q2"}, "id": "t1", "type": "tool_call"}]),
    AIMessage(content="JX48Q2 is cancelled. Rebooking is free for Gold tier."),
])

agent = create_agent(model, tools=[lookup_pnr], system_prompt="You are TravelMind.")

for chunk in agent.stream({"messages": [{"role": "user", "content": "status JX48Q2"}]}, stream_mode="updates"):
    for node, update in chunk.items():
        for m in update.get("messages", []):
            label = type(m).__name__
            detail = f"calls {m.tool_calls[0]['name']}" if getattr(m, "tool_calls", None) else m.content
            print(f"[{node:6}] {label:12} -> {detail}")

[model ] AIMessage    -> calls lookup_pnr
[tools ] ToolMessage  -> BLR-DEL cancelled, Rao, Gold tier
[model ] AIMessage    -> JX48Q2 is cancelled. Rebooking is free for Gold tier.


> **What just happened** the same run, but you saw it unfold: a `model` update requesting the tool, a `tools` update with the record, a final `model` update with the answer. In a real app you would render these as "looking up your booking...", then the answer. The user sees motion instead of a spinner.

---
## What you can now do

- Read an agent transcript as a list of typed messages and spot the tool-call handshake.
- Build a bare agent and a single-tool agent with `create_agent`.
- Stream the loop to see each node update as it happens.
- Swap the scripted model for a live Bedrock model with one line.

**Next, Notebook 3.** We push on tools: multiple tools, how the model chooses between them, what the tool schema actually looks like to the model, and why one bad docstring quietly ruins everything.

> **Skeptic's corner to carry forward** the framework did not make the model smarter. It removed the plumbing around the model. Keep separating those two things. It keeps your expectations honest.